# 03 — Harmonize ses önbelleği (Faz 1)

**Bilimsel amaç:** Her kaydı **bir kez** ve deterministik olarak, bütün modellerin aynı biçimde göreceği hâle getirmek (D-008, D-015, D-018, D-021, D-030). Adımlar:
1. Kayıpsız çözme.
2. Ortak 48 kHz.
3. Kenar sessizliğini kırpma (0.10 s pay bırakılır).
4. 32 kHz ve 16 kHz'e indirme.
5. 32 kHz yolunda 11.0 kHz alçak geçiren filtre.
6. Tepe normalizasyonu (−1 dBFS).

Gürültü giderme **yok**. Pencereleme (4 s / 2 s) burada değil, eğitimde yapılır.

**Bu notebook model EĞİTMEZ.** GPU gerekmez: *Runtime → Change runtime type → CPU*. Süre ~10–20 dk.

**Ön koşullar:**
- Drive'da `data_raw/soundData.zip` ve `data_derived/recording_map.csv` (01 notebook).
- Drive'da **~5 GB boş alan**: 32 kHz ≈ 3.1 GB, 16 kHz ≈ 1.5 GB, float32 (D-018).

**Çıktılar:**
- Drive `data_derived/audio_cache_v1/`: `audio_32k.f32`, `audio_16k.f32`, `index.csv`, `cache_info.json`. Katılımcı düzeyi veri → git'e girmez.
- Repo `reports/audio_cache/`: yalnız agrega rapor. Son hücre bunu ayrı bir dala push eder.

**Gizlilik:** Hiçbir hücre katılımcı ID'si yazdırmaz. Dinleme örnekleri yalnız Drive'a yazılır, notebook'a gömülmez.

## 0. Oturum kurulumu

In [ ]:
import sys, platform, subprocess
import numpy, scipy, pandas
print("Python:", sys.version.split()[0], "|", platform.platform())
print("numpy", numpy.__version__, "| scipy", scipy.__version__, "| pandas", pandas.__version__)
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
DRIVE_ROOT  = Path('/content/drive/MyDrive/asthma-voice')
RAW         = DRIVE_ROOT / 'data_raw'
DERIVED     = DRIVE_ROOT / 'data_derived'
AUDIO_ZIP   = RAW / 'soundData.zip'
AUDIO_DIR   = Path('/content/audio')                 # zip YEREL diske açılır
LOCAL_CACHE = Path('/content/audio_cache_v1')        # önce yerelde üretilir
DRIVE_CACHE = DERIVED / 'audio_cache_v1'             # sonra Drive'a kopyalanır
GITHUB_USER = 'alperarslan19'
REPO_NAME   = 'asthma-voice-dl'
REPO_DIR    = Path('/content') / REPO_NAME

for p in [AUDIO_ZIP, DERIVED / 'recording_map.csv']:
    assert p.exists(), f'Bulunamadı: {p}'
print('Girdiler mevcut.')

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
if not REPO_DIR.exists():
    r = subprocess.run(['git', 'clone', url, str(REPO_DIR)], capture_output=True, text=True)
else:
    r = subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], capture_output=True, text=True)
print((r.stdout + r.stderr).replace(token, '***'))
assert r.returncode == 0, 'git başarısız'
print('Repo commit:', subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip())

In [ ]:
!pip install -q -r {REPO_DIR}/requirements-colab.txt

## 1. Script testi (sahte kayıtlarla, ~30 s)
Bilinen özellikleri olan sahte `.m4a` dosyalarıyla şunları doğrular:
- kırpma miktarı,
- tepe değeri,
- 11 kHz filtresi,
- 44.1 kHz dosya,
- belirlenimcilik,
- değişmiş dosyanın yakalanması.

`Tüm kontroller geçti` görmeden devam etme.

In [ ]:
!cd {REPO_DIR} && python tests/test_build_audio_cache.py

## 2. Ses zip'ini yerel diske kopyala ve aç

In [ ]:
import hashlib, shutil, zipfile, time
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

t0 = time.time()
local_zip = Path('/content/soundData.zip')
if not local_zip.exists():
    shutil.copy(AUDIO_ZIP, local_zip)
print('soundData.zip sha256:', sha256(local_zip), f'({time.time() - t0:.0f} s)')
if not AUDIO_DIR.exists():
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(AUDIO_DIR)
print('Açılan .m4a sayısı:', sum(1 for _ in AUDIO_DIR.rglob('*.m4a')))

## 3. Drive alanı ve üzerine yazma kontrolü
Önbellek bir kez üretilir ve deney boyunca **değişmez**. Drive'da zaten bir önbellek varsa bu hücre durur. Yeniden üretmek bilinçli bir karar olmalı: `OVERWRITE = True` yap ve nedenini EXPERIMENTS.md'ye yaz.

In [ ]:
OVERWRITE = False
need_gb = 5.0
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
print(f'Drive boş alan: {free_gb:.1f} GB (gereken ~{need_gb} GB)')
assert free_gb > need_gb, 'Drive alanı yetmiyor — bana yaz (seçenek: int16 depolama, D-018 değişikliği gerekir)'
if (DRIVE_CACHE / 'cache_info.json').exists():
    assert OVERWRITE, f'{DRIVE_CACHE} zaten var; üzerine yazılmadı.'
print('Devam edilebilir.')

## 4. Önbelleği üret (yerel diske)
Her 200 kayıtta ilerleme yazar. Sonunda doğrulama yapılır:
- kaynak dosyaların sha256'ı denetimle aynı mı;
- NaN/Inf var mı;
- tepe değerleri hedefte mi;
- ofsetler bitişik mi;
- ilk 20 kaydın yeniden işlenince birebir aynı çıkıp çıkmadığı.

Ardından rapor yazdırılır.

In [ ]:
!cd {REPO_DIR} && python scripts/build_audio_cache.py --audio-root {AUDIO_DIR} --recording-map {DERIVED}/recording_map.csv --out-dir {LOCAL_CACHE} --report-dir {REPO_DIR}/reports/audio_cache

## 5. Drive'a kopyala ve sha256 ile doğrula

In [ ]:
import json
info = json.loads((LOCAL_CACHE / 'cache_info.json').read_text())
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
for name in ['audio_32k.f32', 'audio_16k.f32', 'index.csv', 'cache_info.json']:
    t0 = time.time()
    shutil.copy(LOCAL_CACHE / name, DRIVE_CACHE / name)
    if name in info['files_sha256']:
        assert sha256(DRIVE_CACHE / name) == info['files_sha256'][name], f'{name}: Drive kopyası bozuk'
    print(f'{name}: kopyalandı ve doğrulandı ({time.time() - t0:.0f} s)')

## 6. Dinleme kontrolü (Drive'a yazılır, notebook'a gömülmez)
Rastgele 3 kayıt ve kırpma sonrası en kısa kayıt için iki sürüm yazılır:
- orijinalin çözülmüş hâli;
- önbellekteki 32 kHz sürüm.

Drive'da `data_derived/listening_check/` altında dinle. Kontrol et:
- kırpma konuşmanın başını veya sonunu kesmiş mi;
- filtre veya normalizasyon duyulur bir bozulma yaratmış mı.

Hangi örneğin hangi kayıt olduğu yalnız aynı klasördeki `examples.csv`'de durur.

In [ ]:
import numpy as np, pandas as pd, soundfile as sf
sys.path.insert(0, str(REPO_DIR / 'scripts'))
from audit_audio import decode
idx = pd.read_csv(LOCAL_CACHE / 'index.csv')
kept = idx.trim_end_s - idx.trim_start_s
pick = list(idx.sample(3, random_state=0).index) + [int(kept.idxmin())]
out = DERIVED / 'listening_check'
out.mkdir(exist_ok=True)
a32 = np.memmap(LOCAL_CACHE / 'audio_32k.f32', dtype='<f4', mode='r')
rows = []
for j, i in enumerate(pick, 1):
    r = idx.loc[i]
    x = decode(AUDIO_DIR / r.relpath, int(r.src_sr), 1).mean(axis=1)
    sf.write(out / f'ornek{j}_orijinal.wav', x, int(r.src_sr))
    sf.write(out / f'ornek{j}_onbellek_32k.wav', np.asarray(a32[r.offset_32k:r.offset_32k + r.n_32k]), 32000)
    rows.append({'ornek': j, 'relpath': r.relpath, 'kirpma_bas_s': r.lead_trimmed_s, 'kirpma_son_s': r.trail_trimmed_s})
pd.DataFrame(rows).to_csv(out / 'examples.csv', index=False)
print(f'{len(pick)} örnek çifti yazıldı → {out} (4. örnek: kırpma sonrası en kısa kayıt)')

## 7. Raporu GitHub'a gönder (ayrı dal)
Yalnızca `reports/audio_cache/` commit edilir. Bu klasörde agrega sayılar ve sha256'lar var; katılımcı ID'si yok. Sonra yazdırılan bağlantıdan PR'ı aç.

**Notebook'u GitHub'a kaydederken:** çıktılarda katılımcı ID'si olmadığını yine de gözden geçir (D-014).

In [ ]:
BRANCH = 'audio-cache-report'
def git(*a):
    r = subprocess.run(['git', '-C', str(REPO_DIR), *a], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, '***').strip())
    return r
git('config', 'user.name', 'Alper')
git('config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com')
git('checkout', '-B', BRANCH)
git('add', 'reports/audio_cache')
staged = git('diff', '--cached', '--name-only').stdout.split()
assert staged and all(s.startswith('reports/audio_cache/') for s in staged), f'Beklenmeyen dosya: {staged}'
assert git('commit', '-m', 'audio-cache: önbellek raporu (Colab, D-030)').returncode == 0
assert git('push', f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git', BRANCH).returncode == 0
print(f'PR aç: https://github.com/{GITHUB_USER}/{REPO_NAME}/pull/new/{BRANCH}')